# Validation: MegaDetector + BioCLIP vs. trained Faster R-CNN

Runs the new pipeline over the YNP-BisonGraze validation split and compares against ground-truth annotations. The same procedure can be run against Faster R-CNN's `_boxes.npy` / `_scores.npy` outputs (loaded below) to establish a baseline.

**Outputs**: detection precision/recall (IoU ≥ 0.5), classification accuracy on matched detections, and per-class confusion matrix.

In [ ]:
import sys, os, json
from pathlib import Path
from collections import defaultdict, Counter

# `helpers/` has no pyproject; add the repo root so we can import it.
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
from dotenv import load_dotenv

from wytrap.run import process_folder       # requires `pip install -e ../wytrap`
from wytrap.io import load_record
from helpers.helpers import DEFAULT_CATEGORY_MERGES

load_dotenv()
root = os.environ.get("ROOT")
research_project = "YNP-BisonGraze_MC"

val_json_path = os.path.join(root, "annotations", research_project, "clean", "val.json")
image_folder = os.path.join(root, "annotations", research_project, "images")
pipeline_out = os.path.join(root, research_project, "output_pipeline")
frcnn_out    = os.path.join(root, research_project, "output")  # existing FRCNN outputs

## 1. Run the new pipeline over the validation images

Uses the `ynp_testbed` species list so BioCLIP's labels collapse cleanly into the 13 merged categories from `helpers/helpers.py`. Set `resume=True` so re-running this cell only processes new images.

In [ ]:
with open(val_json_path) as f:
    val_coco = json.load(f)

val_image_names = sorted({im["file_name"] for im in val_coco["images"]})
print(f"{len(val_image_names)} validation images")

# Optional: stage val images into a flat folder for processing.
# Or just point process_folder at the full image_folder and filter results.
summary = process_folder(
    input_dir=image_folder,
    output_dir=pipeline_out,
    species="ynp_testbed",
    device="auto",
    resume=True,
    merges=DEFAULT_CATEGORY_MERGES,
)
summary

## 2. Build ground-truth box index

In [ ]:
id_to_name = {c["id"]: c["name"] for c in val_coco["categories"]}
im_id_to_file = {im["id"]: im["file_name"] for im in val_coco["images"]}
gt_by_file = defaultdict(list)  # file_name -> list of (xyxy, label)
for ann in val_coco["annotations"]:
    fname = im_id_to_file[ann["image_id"]]
    if fname.startswith("."):
        continue   # AppleDouble / hidden metadata stubs
    x, y, w, h = ann["bbox"]
    label = id_to_name[ann["category_id"]]
    gt_by_file[fname].append(([int(x), int(y), int(x+w), int(y+h)], label))

print(f"{sum(len(v) for v in gt_by_file.values())} GT boxes across {len(gt_by_file)} images")

## 3. IoU-match predictions to ground truth, score detection P/R + classification accuracy

In [ ]:
def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0

IOU_THRESH = 0.5

def evaluate(gt_by_file, pred_by_file):
    tp = fp = fn = 0
    correct_cls = matched = 0
    confusion = defaultdict(Counter)  # gt_label -> Counter(pred_label)
    for fname, gts in gt_by_file.items():
        preds = pred_by_file.get(fname, [])
        gt_used = [False] * len(gts)
        for pbox, plabel in preds:
            best_iou, best_j = 0, -1
            for j, (gbox, _glabel) in enumerate(gts):
                if gt_used[j]:
                    continue
                v = iou(pbox, gbox)
                if v > best_iou:
                    best_iou, best_j = v, j
            if best_iou >= IOU_THRESH:
                tp += 1
                gt_used[best_j] = True
                glabel = gts[best_j][1]
                matched += 1
                if plabel == glabel:
                    correct_cls += 1
                confusion[glabel][plabel] += 1
            else:
                fp += 1
        fn += sum(1 for u in gt_used if not u)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    cls_acc   = correct_cls / matched if matched else 0.0
    return {"precision": precision, "recall": recall, "cls_acc": cls_acc,
            "tp": tp, "fp": fp, "fn": fn, "matched": matched,
            "confusion": confusion}

# Build pred_by_file from pipeline JSON outputs
pipeline_preds = defaultdict(list)
for jf in Path(pipeline_out).rglob("*.json"):
    rec = load_record(jf)
    fname = Path(rec.image_path).name
    if fname.startswith("."):
        continue
    if fname not in gt_by_file:
        continue
    for det in rec.detections:
        pipeline_preds[fname].append((det.box_xyxy, det.label))

pipe_metrics = evaluate(gt_by_file, pipeline_preds)
print("=== MegaDetector + BioCLIP ===")
for k in ("precision", "recall", "cls_acc", "tp", "fp", "fn", "matched"):
    print(f"  {k}: {pipe_metrics[k]}")

## 4. Baseline: same evaluation against the trained Faster R-CNN outputs

Loads the existing `_boxes.npy` files written by `inference.ipynb`. The trained model doesn't currently emit class labels in the saved files, so this evaluates **detection only**. Drop this cell if you haven't run the FRCNN inference notebook yet.

In [ ]:
frcnn_preds = defaultdict(list)
for boxes_npy in Path(frcnn_out).glob("*_boxes.npy"):
    stem = boxes_npy.name.replace("_boxes.npy", "")
    fname = stem + ".JPG"  # adjust extension if needed
    if fname not in gt_by_file:
        continue
    boxes = np.load(boxes_npy)
    scores_npy = boxes_npy.with_name(stem + "_scores.npy")
    scores = np.load(scores_npy) if scores_npy.exists() else np.ones(len(boxes))
    for box, score in zip(boxes, scores):
        if score < 0.5:
            continue
        frcnn_preds[fname].append(([int(v) for v in box], "<unlabeled>"))

frcnn_metrics = evaluate(gt_by_file, frcnn_preds)
print("=== Faster R-CNN (detection only) ===")
for k in ("precision", "recall", "tp", "fp", "fn"):
    print(f"  {k}: {frcnn_metrics[k]}")

## 5. Per-class confusion matrix for the new pipeline

In [ ]:
labels_in_use = sorted(set(pipe_metrics["confusion"].keys()) | {
    p for c in pipe_metrics["confusion"].values() for p in c.keys()
})
print(f"{'gt\\pred':<20} " + " ".join(f"{l[:8]:>8}" for l in labels_in_use))
for gt in labels_in_use:
    row = pipe_metrics["confusion"].get(gt, Counter())
    print(f"{gt:<20} " + " ".join(f"{row.get(p, 0):>8}" for p in labels_in_use))